# Lesson 5 — `pragma_mini.py`, executable line by line

This notebook is a runnable version of the [Lesson 5 walkthrough](../05_putting_it_together.md).
Each section has a **markdown explanation** followed by **executable code**.

Run cells top to bottom. Tweak anything. Tweak it back. The point is to
*feel* the code under your fingers.

## 🧰 Lesson reference legend

- **L1** — the 5-line training loop (predict → loss → backward → step)
- **L1b** — architecture vs. training
- **L1c** — gradient descent details
- **L2** — tokens & embeddings
- **L3** — attention
- **L3b** — why Transformers won
- **L4** — masked language modelling


## 0 — Imports and seeds

Set a fixed random seed so every run produces the same numbers. Makes
debugging easier.


In [ ]:
import random                              # Python's stdlib random (for random_event).
import torch                                # PyTorch: tensors, autograd, neural nets.
import torch.nn as nn                       # nn.Linear, nn.Embedding, nn.Module, ...

torch.manual_seed(0)                        # Reproducible PyTorch RNG.
random.seed(0)                              # Reproducible Python RNG.

print("torch", torch.__version__)

## 1 — Vocabulary (L2)

A vocabulary list (14 tokens) plus a dict mapping each token to its ID.
Standard tokenisation — same setup as Lesson 2.

The new twist: tokens fall into two groups.

- **KEYS** are field names ("what kind of fact am I about to tell you?").
- **VALUES** are the actual contents.

This is the *key-value tokenisation* PRAGMA uses for structured data —
see Lesson 5 walkthrough for the deep explanation.


In [ ]:
KEYS   = ["pet", "action", "place"]         # Field names — 3 of them.
VALUES = ["dog", "cat", "fish",             # Pet values.
          "eat", "sleep", "play",            # Action values.
          "garden", "couch", "bowl"]         # Place values.

PAD, MASK = "<pad>", "<mask>"                # Special tokens (padding & masking).
vocab = [PAD, MASK] + KEYS + VALUES          # Full vocab: specials + keys + values.
tok2id = {t: i for i, t in enumerate(vocab)} # Lookup table: token → id.
V = len(vocab)                               # Vocabulary size (14 tokens here).

print("vocab:", vocab)
print("V =", V)

## 2 — Synthetic data (the secret rules)

Plain Python — no ML yet. `RULES` defines the hidden patterns the model
will discover from training. **The model never sees `RULES` directly.**
It only sees events that obey them, and has to back out the structure.

Notice: dogs play in the garden, fish only go in the bowl, etc.


In [ ]:
RULES = {                                   # The hidden patterns. Model never sees this dict.
    "dog":  {"action": ["eat", "play"],   "place": ["garden", "bowl"]},
    "cat":  {"action": ["sleep", "play"], "place": ["couch", "bowl"]},
    "fish": {"action": ["eat", "sleep"],  "place": ["bowl"]},
}

def random_event():                         # Sample one event respecting the RULES.
    pet = random.choice(list(RULES))        # Pick a random pet.
    act = random.choice(RULES[pet]["action"])   # Pick a valid action for that pet.
    plc = random.choice(RULES[pet]["place"])    # Pick a valid place for that pet.
    return [("pet", pet), ("action", act), ("place", plc)]    # 3 (key, value) pairs.

# Generate a few samples
for _ in range(5):
    print(random_event())

## 3 — Encoding an event (L2)

Turn a list of `(key, value)` pairs into a flat list of token IDs.


In [ ]:
def encode(event):                          # Flatten event into a list of token ids.
    ids = []
    for k, v in event:                      # Iterate over (key, value) pairs.
        ids.append(tok2id[k])               # Append the key's token id.
        ids.append(tok2id[v])               # Append the value's token id.
    return ids                              # Returns a flat list — 6 ids for our 3-pair events.

# Example
event = [("pet", "dog"), ("action", "play"), ("place", "garden")]
print("event:  ", event)
print("encoded:", encode(event))
print("decoded back:", [vocab[i] for i in encode(event)])      # Roundtrip check.

## 4 — Masking (L4)

The fill-in-the-blank game. For each value token (not key tokens — we
want to keep the field name as a hint), flip a 25% coin. If heads:
remember the truth, replace with `<mask>`.

`labels[i] = -100` for unmasked positions tells the loss function
"don't grade this position". We only score on the holes we made.


In [ ]:
def mask(ids, p=0.25):                      # Randomly mask value tokens with probability p.
    ids = list(ids)                         # Copy so we can mutate.
    labels = [-100] * len(ids)              # -100 = "skip this position in loss".
    for i, t in enumerate(ids):
        if vocab[t] in KEYS:                # Never mask KEY tokens — they're the prompt.
            continue
        if random.random() < p:             # Coin flip: mask this value?
            labels[i] = t                   # Remember the truth at this position.
            ids[i] = tok2id[MASK]           # Replace with <mask>.
    return ids, labels

# Try it
event = [("pet", "dog"), ("action", "play"), ("place", "garden")]
masked, labels = mask(encode(event), p=0.5)         # Higher prob so we see something masked.
print("input  :", [vocab[i] for i in masked])
print("labels :", labels, "  (-100 = ignored, real id = answer to predict)")

## 5 — The architecture (L1b + L2 + L3 + L3b)

Six lines define the whole Transformer.

| Piece | Lesson | What it does |
|-------|--------|--------------|
| `nn.Embedding(V, d)` | **L2** | Token embedding table (14 × 32 = 448 knobs) |
| `nn.Embedding(64, d)` | (positional) | Position embeddings |
| `TransformerEncoder(layer, layers=2)` | **L3** | Two attention+FFN layers stacked |
| `nn.Linear(d, V)` | (MLM head) | Project context vectors back to vocab scores |

Real PRAGMA-L stacks **18** of those encoder layers. Each one runs
attention again on the previous layer's output, building progressively
richer understanding.


In [ ]:
class PragmaMini(nn.Module):
    def __init__(self, V, d=32, heads=2, layers=2):     # V=vocab, d=dim, heads/layers config.
        super().__init__()                              # Required nn.Module boilerplate.
        self.emb = nn.Embedding(V, d)                   # Token → vector lookup. V × d knobs.
        self.pos = nn.Embedding(64, d)                  # Position embeddings (up to 64 positions).
        layer = nn.TransformerEncoderLayer(d, heads, 64, batch_first=True)  # One block template.
        self.enc = nn.TransformerEncoder(layer, layers) # Stack `layers` copies of the block.
        self.head = nn.Linear(d, V)                     # Project back to vocab scores (MLM head).
    def forward(self, x):                               # x shape: (B, L) — batch of token id seqs.
        positions = torch.arange(x.size(1), device=x.device)    # [0, 1, ..., L-1].
        h = self.enc(self.emb(x) + self.pos(positions))         # Combine token + position, then encode.
        return self.head(h)                             # Vocab scores per position. Shape (B, L, V).

model = PragmaMini(V)                                   # Instantiate with random initial weights.
print(f"Total knobs: {sum(p.numel() for p in model.parameters()):,}")

## 6 — Training (L1 + L1c + L4)

The 5-line training loop you've now seen many times. The only
differences from Lesson 1's `w, b` regression:

- `AdamW` instead of SGD — fancier optimiser, same principle.
- `CrossEntropyLoss` instead of MSE — picks 1 of `V` words.
- ~14,000 knobs being nudged instead of 2.

Should take 30 seconds or so on CPU.


In [ ]:
opt     = torch.optim.AdamW(model.parameters(), lr=3e-3)     # AdamW optimiser.
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)              # CE that skips -100 labels.

print("Training (2000 steps)...")
for step in range(2000):
    batch = [random_event() for _ in range(32)]              # 32 random events per batch.
    masked, labels = zip(*[mask(encode(e)) for e in batch])  # Encode + mask each event.
    x = torch.tensor(masked)                                 # Stack masked inputs.
    y = torch.tensor(labels)                                 # Stack label arrays.

    logits = model(x)                                    # 1. predict
    loss = loss_fn(logits.reshape(-1, V), y.reshape(-1)) # 2. measure
    opt.zero_grad()                                      # 3. clear
    loss.backward()                                      # 4. backward
    opt.step()                                           # 5. nudge

    if step % 400 == 0:
        print(f"  step {step:4d}   loss {loss.item():.3f}")

print("Done.")

## 7 — Inference (play with the trained model)

Hide one field, ask the model to fill the blank.


In [ ]:
def guess(event, hide_key):                 # Mask the value of `hide_key`, predict it.
    ids = encode(event)                     # Encode the full event.
    pos = None
    for i in range(0, len(ids), 2):         # Walk key positions (even indices: 0, 2, 4...).
        if vocab[ids[i]] == hide_key:       # Found the key we want to mask?
            pos = i + 1                     # The value is at the next position.
            ids[pos] = tok2id[MASK]         # Mask it.
    with torch.no_grad():
        logits = model(torch.tensor([ids])) # Forward pass on the masked input.
    return vocab[logits[0, pos].argmax().item()]    # Decode argmax to vocab token name.

print("dog is playing in the ____ ->",
      guess([("pet","dog"), ("action","play"), ("place","garden")], "place"))
print("cat is sleeping on the ____ ->",
      guess([("pet","cat"), ("action","sleep"), ("place","couch")], "place"))
print("fish is eating in the ____ ->",
      guess([("pet","fish"), ("action","eat"), ("place","bowl")], "place"))
print("____ is sleeping on the couch ->",
      guess([("pet","dog"), ("action","sleep"), ("place","couch")], "pet"))
print("dog is ____ in the garden ->",
      guess([("pet","dog"), ("action","eat"), ("place","garden")], "action"))

## 8 — Things to try

Now that everything's loaded, experiment freely. Some ideas:

1. **Train less.** Change `range(2000)` to `range(200)`. Re-run. Predictions get worse.
2. **Bigger model.** In the `PragmaMini` constructor, set `layers=4` or `d=64`. Re-train. More knobs = better representation.
3. **Add a new pet.** Add a parrot to `RULES` with its own action/place set. Re-build the vocab and re-train.
4. **Inspect the embedding table.** `model.emb.weight` is a 14×32 tensor of trained numbers. Compare `model.emb.weight[tok2id["dog"]]` with `[tok2id["cat"]]` — after training, similar pets should have similar vectors.

## What's next

You've just executed PRAGMA's recipe end-to-end. Next up:

- **[Lesson 5b notebook](lesson_05b_streaming_churn.ipynb)** — same recipe, more realistic problem (predicting streaming-service churn).
- **[Lesson 6 — Capstone](../06_capstone.md)** — build your own version on fraud detection.
